# CADeT Listwise Distillation — Colab Training Run

**Goal:** fine-tune `ManmohanBuildsProducts/auto-parts-search-v3` on the listwise teacher-scored dataset + golden-v2 InfoNCE, push result to `ManmohanBuildsProducts/auto-parts-search-v4-cadet`.

**Requirements:** T4 GPU (free) or better. Runs in ~1-2 hrs on T4.

**Before running:** Runtime → Change runtime type → GPU (T4).

## 1. Install dependencies

In [ ]:
!pip install -q 'sentence-transformers>=2.2.0' 'transformers>=4.48.0,<5.0.0' 'huggingface_hub>=0.24.0,<1.0.0' 'datasets>=2.14.0'

## 2. HF login (paste your token from https://huggingface.co/settings/tokens — needs write access)

In [ ]:
from huggingface_hub import login
login()  # prompts for token

## 3. Verify GPU

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

## 4. Inline training script (same logic as `training/train_listwise.py`)

In [ ]:
import argparse, json, os, torch
import torch.nn.functional as F
from itertools import cycle
from pathlib import Path
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, Dataset as TorchDataset


# --- Listwise KL + InfoNCE loss ---
def compute_listwise_kl(query_emb, doc_embs, teacher_scores, temperature=1.0):
    student_logits = torch.bmm(doc_embs, query_emb.unsqueeze(-1)).squeeze(-1) / temperature
    student_log_probs = F.log_softmax(student_logits, dim=-1)
    teacher_probs = F.softmax(teacher_scores / temperature, dim=-1)
    return F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')


def compute_infonce(query_emb, doc_embs, temperature=0.05):
    logits = torch.bmm(doc_embs, query_emb.unsqueeze(-1)).squeeze(-1) / temperature
    targets = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
    return F.cross_entropy(logits, targets)


class ListwiseKLLoss(torch.nn.Module):
    def __init__(self, lambda_listwise=0.6, lambda_infonce=0.4):
        super().__init__()
        self.lambda_listwise = lambda_listwise
        self.lambda_infonce = lambda_infonce

    def forward(self, q, d, t):
        return self.lambda_listwise * compute_listwise_kl(q, d, t) + self.lambda_infonce * compute_infonce(q, d)

## 5. Dataset loaders

In [ ]:
LISTWISE_DATASET = 'ManmohanBuildsProducts/auto-parts-listwise-v1'
GOLDEN_DATASET = 'ManmohanBuildsProducts/auto-parts-search-pairs'

print('loading listwise dataset...')
listwise_ds = load_dataset(LISTWISE_DATASET, split='train')
print(f'  {len(listwise_ds)} records')

print('loading golden-v2 dataset...')
golden_ds = load_dataset(GOLDEN_DATASET, split='train')
print(f'  {len(golden_ds)} pairs (filtering to label=1.0...)')


class ListwiseDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.records = list(hf_dataset)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        cand_titles = json.loads(rec['candidate_doc_titles'])
        teacher_scores = json.loads(rec['teacher_scores'])
        gold_title = rec['gold_doc_title']
        if gold_title in cand_titles:
            i = cand_titles.index(gold_title)
            cand_titles.insert(0, cand_titles.pop(i))
            teacher_scores.insert(0, teacher_scores.pop(i))
        else:
            cand_titles.insert(0, gold_title)
            teacher_scores.insert(0, 1.0)
        return {
            'query': rec['query'],
            'candidates': cand_titles[:20],
            'teacher_scores': torch.tensor(teacher_scores[:20], dtype=torch.float),
        }


class GoldenV2Dataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.records = [
            {'query': r['text_a'], 'positive': r['text_b']}
            for r in hf_dataset
            if r.get('text_a') and r.get('text_b') and r.get('label') == 1.0
        ]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        return self.records[idx]


gv2 = GoldenV2Dataset(golden_ds)
print(f'  golden-v2 usable pairs (label=1.0): {len(gv2)}')

## 6. Load v3 base model

In [ ]:
BASE_MODEL = 'ManmohanBuildsProducts/auto-parts-search-v3'
MAX_SEQ_LEN = 128

print(f'loading {BASE_MODEL}...')
model = SentenceTransformer(BASE_MODEL, trust_remote_code=True)
model.max_seq_length = MAX_SEQ_LEN
model = model.to('cuda')
model[0].auto_model.gradient_checkpointing_enable()
print('model loaded on CUDA with gradient checkpointing')

## 7. Train

In [ ]:
EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5
GOLDEN_INTERLEAVE_RATIO = 2
CKPT_DIR = Path('/content/checkpoints/cadet')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
device = 'cuda'


def encode(model, texts, device):
    features = model.tokenize(texts)
    features = {k: v.to(device) for k, v in features.items()}
    out = model(features)
    return F.normalize(out['sentence_embedding'], p=2, dim=-1)


criterion = ListwiseKLLoss(lambda_listwise=0.6, lambda_infonce=0.4)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
n_steps = (len(listwise_ds) // BATCH_SIZE + len(listwise_ds) // BATCH_SIZE // GOLDEN_INTERLEAVE_RATIO) * EPOCHS
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.0, total_iters=max(n_steps, 1))

best_loss = float('inf')
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    listwise_loader = DataLoader(ListwiseDataset(listwise_ds), batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: x)
    golden_loader = cycle(DataLoader(gv2, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: x))

    for step, batch in enumerate(listwise_loader):
        queries = [item['query'] for item in batch]
        all_candidates = [item['candidates'] for item in batch]
        all_teacher = torch.stack([item['teacher_scores'] for item in batch]).to(device)

        q_emb = encode(model, queries, device)
        flat = [c for cands in all_candidates for c in cands]
        flat_emb = encode(model, flat, device)
        k = len(all_candidates[0])
        doc_embs = flat_emb.view(len(batch), k, -1)

        loss = criterion(q_emb, doc_embs, all_teacher)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()
        n_batches += 1

        if step % GOLDEN_INTERLEAVE_RATIO == 0:
            gbatch = next(golden_loader)
            gq = encode(model, [it['query'] for it in gbatch], device)
            gp = encode(model, [it['positive'] for it in gbatch], device)
            scores = torch.mm(gq, gp.t()) / 0.05
            targets = torch.arange(len(gbatch), device=device)
            g_loss = F.cross_entropy(scores, targets)
            optimizer.zero_grad()
            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += g_loss.item()
            n_batches += 1

        if step % 20 == 0:
            print(f'  epoch {epoch+1} step {step}/{len(listwise_loader)} — loss {loss.item():.4f}')

    avg = epoch_loss / max(n_batches, 1)
    print(f'epoch {epoch+1}/{EPOCHS} — avg loss: {avg:.4f}')
    if avg < best_loss:
        best_loss = avg
        model.save(str(CKPT_DIR / 'best'))
        print(f'  saved best (loss={best_loss:.4f})')

## 8. Push to HF Hub

In [ ]:
OUTPUT_REPO = 'ManmohanBuildsProducts/auto-parts-search-v4-cadet'
best = SentenceTransformer(str(CKPT_DIR / 'best'), trust_remote_code=True)
best.push_to_hub(OUTPUT_REPO, private=True)
print(f'pushed → https://huggingface.co/{OUTPUT_REPO}')